# Pipeline Layer Demo

This notebook walks through the scripts/apis_pipe pipeline layer.

4 main components of the pipeline:

SpeciesAPI
-- scripts/apis_pipe/base.py
-- Abstract base class all API clients must implement 

TaxonRouter
-- scripts/utils/router.py
-- Asks GBIF for the kingdom, returns the right set of APIs to query

SynonymEngine
-- scripts/utils/synonyms.py
-- Fetches synonyms from backbone APIs and normalizes them

SpeciesAggregator
-- scripts/utils/aggregator.py
-- Fan-out occurrence search across all selected APIs

The top-level entry point call_apis() in scripts/utils/call_apis_pipe.py runs all 4 of them together. 

In [7]:
import json
import sys
from pathlib import Path

# Make sure the project root is on the path when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

QUERY = "Amanita muscaria"

### 1. TaxonRouter for kingdom detection and API selection

`TaxonRouter` makes one GBIF lookup to find the kingdom, then returns the appropriate list of API keys to query.
This avoids querying plant-only databases (e.g. Tropicos) for fungi, and vice versa.

In [8]:
from scripts.apis_pipe.gbif import GBIFAPI
from scripts.utils.router import TaxonRouter

gbif_client = GBIFAPI()
router = TaxonRouter(gbif_client=gbif_client)

kingdom = router._get_kingdom(QUERY)
print(f"Kingdom: {kingdom}")

Kingdom: Fungi


In [9]:
selected_apis = router.route(QUERY)
print(f"APIs selected for '{QUERY}':")
for api in selected_apis:
    print(f"  {api}")

APIs selected for 'Amanita muscaria':
  gbif
  col
  genbank
  index_fungorum
  mushroomobs
  symbiota_mycoportal
  symbiota_lichen


### 2. SynonymEngine — normalized synonym records

`SynonymEngine` queries backbone APIs (GBIF, COL, Tropicos, Index Fungorum) and wraps each result into a standardized dict with `name`, `author`, `date`, `publishedIn`, `url`, `source`, and `confidence`.
Results are deduplicated by canonical name before being returned.

In [10]:
from scripts.apis_pipe.col import COLAPI
from scripts.apis_pipe.index_fungorum import IndexFungorumAPI
from scripts.apis_pipe.tropicos import TropicosAPI
from scripts.utils.synonyms import SynonymEngine

syn_engine = SynonymEngine(
    gbif=gbif_client,
    tropicos=TropicosAPI(),
    index_fungorum=IndexFungorumAPI(),
    col=COLAPI(),
)

synonyms = syn_engine.get_synonyms(QUERY)
print(f"{len(synonyms)} synonym(s) found")

14 synonym(s) found


In [11]:
import pandas as pd

# Drop the raw provenance field for display
display_cols = ["name", "author", "date", "source", "confidence", "url"]
df_syn = pd.DataFrame(
    [{k: s[k] for k in display_cols} for s in synonyms]
)
df_syn

,name,author,date,source,confidence,url
0,Amanita muscaria,,,GBIF,1.0,https://www.gbif.org/species/8168319
1,Agaricus aureolus,Kalchbr.,,GBIF,0.9,https://www.gbif.org/species/5455639
2,Agaricus imperialis,Batsch,,GBIF,0.9,https://www.gbif.org/species/5955425
3,Agaricus muscarius,L.,,GBIF,0.9,https://www.gbif.org/species/5451774
4,Agaricus nobilis,Bolton,,GBIF,0.9,https://www.gbif.org/species/3329091
5,Agaricus pseudoaurantiacus,Bull.,,GBIF,0.9,https://www.gbif.org/species/3329092
6,Agaricus puellus,Batsch,,GBIF,0.9,https://www.gbif.org/species/5452606
7,Amanita aureola,(Kalchbr.) Sacc.,,GBIF,0.9,https://www.gbif.org/species/5452441
8,Amanita circinnata,Gray,,GBIF,0.9,https://www.gbif.org/species/5452605
9,Amanita formosa,Gonn. & Rabenh.,,GBIF,0.9,https://www.gbif.org/species/5452657


### 3. SpeciesAggregator — occurrence records

`SpeciesAggregator.occurrences()` fans out across all selected APIs, searching for both the primary name and every synonym.
Each API result is isolated in a `try/except`, so a timeout or dead server produces a `"warning"` status rather than crashing the whole pipeline.

In [12]:
from scripts.apis_pipe.genbank import GenBankAPI
from scripts.apis_pipe.mushroomobs import MushroomObserverAPI
from scripts.apis_pipe.symbiota import SymbiotaAPI
from scripts.utils.aggregator import SpeciesAggregator

clients = {
    "gbif": gbif_client,
    "col": COLAPI(),
    "genbank": GenBankAPI(),
    "index_fungorum": IndexFungorumAPI(),
    "mushroomobs": MushroomObserverAPI(),
    "symbiota_mycoportal": SymbiotaAPI("https://mycoportal.org/portal"),
}

aggregator = SpeciesAggregator(clients=clients, router=router)

synonym_names = [s["name"] for s in synonyms if s["name"]]
apis_to_query = [k for k in selected_apis if k in clients]  # limit to what we initialized

occurrences = aggregator.occurrences(
    name=QUERY,
    synonyms=synonym_names,
    apis=apis_to_query,
    limit=5,
)

# Show status summary
for api_key, result in occurrences.items():
    status = result.get("status")
    count = len(result.get("data", []))
    msg = result.get("message", f"{count} record(s)")
    print(f"  {api_key:30s}  {status:8s}  {msg}")

  gbif                            success   65 record(s)
  col                             success   0 record(s)
  genbank                         success   15 record(s)
  index_fungorum                  success   0 record(s)
  mushroomobs                     success   10 record(s)
  symbiota_mycoportal             success   0 record(s)


In [14]:
# Inspect records from GBIF
gbif_records = occurrences.get("gbif", {}).get("data", [])
print(f"{len(gbif_records)} GBIF occurrence record(s)")
if gbif_records:
    print(json.dumps(gbif_records[0], indent=2))

65 GBIF occurrence record(s)
{
  "key": 6061972505,
  "datasetKey": "032f1fbe-d21f-407d-be46-5cd1bb1ea397",
  "publishingOrgKey": "61c539d8-47e0-4b82-875f-b95aad66dcd4",
  "networkKeys": [
    "68e8e67a-43e6-44a3-8817-dc0e6b70f973",
    "0479d6b2-a4f4-4307-8346-0b80e8c88c66"
  ],
  "installationKey": "dd8ffb07-5f02-4e70-b4cd-dbba0dfae83e",
  "hostingOrganizationKey": "96710dc8-fecb-440d-ae3e-c34ae8a9616f",
  "publishingCountry": "US",
  "protocol": "DWC_ARCHIVE",
  "lastCrawled": "2026-05-27T03:27:39.341+00:00",
  "lastParsed": "2026-05-27T03:30:09.881+00:00",
  "crawlId": 374,
  "extensions": {
    "http://rs.tdwg.org/ac/terms/Multimedia": [
      {
        "http://rs.tdwg.org/ac/terms/accessURI": "https://biocoll.inhs.illinois.edu/media/myco/TENN-F/TENN-F-079/Amanita_muscaria_v_guessowii_P_1750249233.jpg",
        "http://ns.adobe.com/xap/1.0/rights/UsageTerms": "P.B. Matheny",
        "http://ns.adobe.com/xap/1.0/rights/Owner": "University of Tennessee Fungal Herbarium (TENN-F-)",
 

### 4. Full pipeline — `call_apis()`

`call_apis()` wires together the router, synonym engine, and aggregator into one call.
The returned JSON has two top-level keys: `"synonyms"` and `"occurrences"`.

In [15]:
from scripts.utils.call_apis_pipe import call_apis

raw = call_apis(QUERY, limit=5)
payload = json.loads(raw)

print("Top-level keys:", list(payload.keys()))
print("Synonym sub-keys:", list(payload["synonyms"].keys()))
print("Sources in occurrences:", list(payload["occurrences"].keys()))

Top-level keys: ['synonyms', 'occurrences']
Synonym sub-keys: ['official', 'symbiota', 'independent']
Sources in occurrences: ['gbif', 'col', 'genbank', 'index_fungorum', 'mushroomobs', 'symbiota_mycoportal', 'symbiota_lichen']


In [16]:
# Official synonyms summary
official = payload["synonyms"]["official"]
df_official = pd.DataFrame(
    [{k: s.get(k, "") for k in display_cols} for s in official if "error" not in s]
)
print(f"{len(df_official)} official synonym(s)")
df_official

14 official synonym(s)


,name,author,date,source,confidence,url
0,Amanita muscaria,,,GBIF,1.0,https://www.gbif.org/species/8168319
1,Agaricus aureolus,Kalchbr.,,GBIF,0.9,https://www.gbif.org/species/5455639
2,Agaricus imperialis,Batsch,,GBIF,0.9,https://www.gbif.org/species/5955425
3,Agaricus muscarius,L.,,GBIF,0.9,https://www.gbif.org/species/5451774
4,Agaricus nobilis,Bolton,,GBIF,0.9,https://www.gbif.org/species/3329091
5,Agaricus pseudoaurantiacus,Bull.,,GBIF,0.9,https://www.gbif.org/species/3329092
6,Agaricus puellus,Batsch,,GBIF,0.9,https://www.gbif.org/species/5452606
7,Amanita aureola,(Kalchbr.) Sacc.,,GBIF,0.9,https://www.gbif.org/species/5452441
8,Amanita circinnata,Gray,,GBIF,0.9,https://www.gbif.org/species/5452605
9,Amanita formosa,Gonn. & Rabenh.,,GBIF,0.9,https://www.gbif.org/species/5452657


In [17]:
# Occurrence status summary
rows = []
for api_key, result in payload["occurrences"].items():
    rows.append({
        "source": api_key,
        "status": result.get("status"),
        "records": len(result.get("data", [])),
        "message": result.get("message", ""),
    })
pd.DataFrame(rows)

,source,status,records,message
0,gbif,success,60,
1,col,success,0,
2,genbank,success,10,
3,index_fungorum,success,0,
4,mushroomobs,success,10,
5,symbiota_mycoportal,success,0,
6,symbiota_lichen,success,0,


Also let's take a look at base.py abstract class, router.py, and synonyms.py if time. 